Why You Should Learn Spatial SQL

In earlier articles, we saw how spatial data is becoming a first-class citizen in the Databricks Lakehouse. One of the core pillars of this transformation is the arrival of 80+ built-in Spatial SQL functions.

These functions allow you to:

- Build geometry from raw data (lat/long, WKT, GeoJSON)
- Perform spatial joins and spatial filtering
- Measure distance, area, and relationships
- Transform and analyze shapes (buffer, simplify, union, convex hull)
- Integrate with multi-modal data (IoT, streaming, sensor, fleet, retail)

Today, we’ll learn the 10 most useful Spatial SQL functions, hands-on.

![](/Volumes/thedataengineering_00/data/sampledata/SpatialDatatypes.png)

Top 10 Spatial Functions You Must Know

Let’s explore the most important functions with runnable examples.

Each example assumes we have created this DataFrame:

In [0]:
from pyspark.sql.functions import expr

data = [
    ("Location A", 40.748817, -73.985428),   # NYC
    ("Location B", 34.052235, -118.243683),  # LA
    ("Location C", 51.507222, -0.1275)       # London
]
df = spark.createDataFrame(data, ["name", "lat", "lon"])

df = df.withColumn("geometry_point", expr("ST_POINT(lon, lat)"))  # GEOMetry Point
df.createOrReplaceTempView("locations")

### 1. ST_Point

Convert latitude and longitude into a real GEOGRAPHY object.

In [0]:
%sql
SELECT name, ST_POINT(lon, lat) AS geo_location
FROM locations;

### 2. ST_Distance

Measure distance between two spatial objects (in meters for GEOGRAPHY)

In [0]:
%sql
SELECT
  a.name AS from_loc,
  b.name AS to_loc,
  ST_DISTANCE(a.geo_point, b.geo_point) AS distance
FROM locations a CROSS JOIN locations b
WHERE a.name = 'Location A' AND b.name = 'Location C';

Lets take example for Geography point with SRID

In [0]:
from pyspark.sql.functions import expr

data = [
    ("Location A", 40.748817, -73.985428),   # NYC
    ("Location B", 34.052235, -118.243683),  # LA
    ("Location C", 51.507222, -0.1275)       # London
]

df = spark.createDataFrame(data, ["name", "lat", "lon"])

# Create a GEOMETRY point and explicitly set SRID: 4326 (WGS84)
df = df.withColumn(
    "geometry",
    expr("ST_SETSRID(ST_POINT(lon, lat), 4326)")
)

df.createOrReplaceTempView("locations")
df.show(truncate=False)


Calculate distance in KM.

In [0]:
%sql
SELECT
  ST_DISTANCE(
    ST_TRANSFORM(a.geometry, 3857),
    ST_TRANSFORM(b.geometry, 3857)
  ) / 1000 AS distance_km
FROM locations a, locations b
WHERE a.name = 'Location A' AND b.name = 'Location B';


### 3. ST_Buffer

Create a buffer around a point or polygon — useful for building catchment zones.

In [0]:
from pyspark.sql.functions import expr

data = [
    ("Location A", 40.748817, -73.985428),   # NYC
    ("Location B", 34.052235, -118.243683),  # LA
    ("Location C", 51.507222, -0.1275),       # London
    ("Location D",21.1125, 79.0465)
]
df = spark.createDataFrame(data, ["name", "lat", "lon"])

df = df.withColumn("geometry_point", expr("ST_POINT(lon, lat)"))  # GEOMetry Point
df.createOrReplaceTempView("locations")

In [0]:
%sql
SELECT
  name,
  ST_BUFFER(geometry_point, 5000) AS area_5km  -- 5000 meters radius
FROM locations
WHERE name = 'Location D';

##Lets Visualize

In [0]:
# Fetch buffer polygon from SQL into a dataframe
buffer_df = spark.sql("""
SELECT
  name,
  ST_ASTEXT(ST_BUFFER(geometry_point, 100)) AS wkt_area_5km
FROM locations
WHERE name = 'Location D'
""").toPandas()

buffer_df


In [0]:
%pip install folium
%pip install shapely

In [0]:
import folium
from shapely import wkt

# Extract geometry
polygon_wkt = buffer_df.loc[0, "wkt_area_5km"]
polygon_geom = wkt.loads(polygon_wkt)
centroid = polygon_geom.centroid

# Create base map
m = folium.Map(location=[centroid.y, centroid.x], zoom_start=10)

# Add polygon (geographic buffer) - disappears when zoomed out
folium.GeoJson(
    data=polygon_geom.__geo_interface__,
    name="Buffer Area",
    style_function=lambda x: {"fillColor": "blue", "color": "blue", "fillOpacity": 0.3},
).add_to(m)

# Add circle overlay (pixel-scaled, always visible)
folium.Circle(
    location=[centroid.y, centroid.x],
    radius=5000,   # same 5 km radius (meters)
    color="red",
    fill=True,
    fill_opacity=0.2,
    popup="5 km Radius",
).add_to(m)

# Add layer control
folium.LayerControl().add_to(m)

m



### 4. ST_Contains
Check if a point lies within a buffer or boundary.

In [0]:
from pyspark.sql.functions import expr

# Sample locations with (latitude, longitude)
data = [
    ("Location A", 40.748817, -73.985428),   # NYC (for buffer)
    ("Location B", 40.731128, -73.997360),   # Greenwich Village, NYC
    ("Location C", 40.761432, -73.977621),   # Central Park, NYC
    ("Location D", 40.730610, -73.935242),   # Queens, NYC
    ("Location E", 40.650002, -73.949997),   # Brooklyn, NYC
    ("Location F", 34.052235, -118.243683)   # Los Angeles (outside buffer)
]

# Create DataFrame
df = spark.createDataFrame(data, ["name", "lat", "lon"])

# Convert to GEOMETRY type with SRID 4326
df = df.withColumn("geo_point", expr("ST_SETSRID(ST_POINT(lon, lat), 4326)"))

# Create or replace table for SQL usage
df.createOrReplaceTempView("locations")
df.show(truncate=False)


In [0]:
%sql
WITH buffered AS (
  SELECT 
    name,
    ST_BUFFER(geo_point, 10000) AS zone
  FROM locations 
  WHERE name = 'Location A'
)
SELECT 
  a.name AS reference_point, 
  b.name AS nearby_point
FROM buffered a
JOIN locations b
WHERE ST_CONTAINS(a.zone, b.geo_point);




This SQL code creates a 10 km radius buffer around "Location A" and then finds all other locations whose geometric point (geo_point) falls inside that buffer zone.

In short:
It identifies which locations are within 10 km of Location A using spatial containment logic.

### 5. ST_Intersects

Return true if two geometries intersect.

In [0]:
from pyspark.sql.functions import expr

data = [
    ("Location A", 40.748817, -73.985428),  # Near Times Square, NYC
    ("Location B", 40.731128, -73.997360),  # Greenwich Village, NYC
    ("Location C", 34.052235, -118.243683)  # Los Angeles (far away)
]

df = spark.createDataFrame(data, ["name", "lat", "lon"])

# Convert to valid geometry with SRID 4326
df = df.withColumn("geo_point", expr("ST_SETSRID(ST_POINT(lon, lat), 4326)"))

df.createOrReplaceTempView("locations")
df.show(truncate=False)


### How It Works

- ST_BUFFER(geo_point, 5000) creates a 5,000-meter (5 km) radius circle around each location.
- ST_INTERSECTS(buffer_A, buffer_B) checks if the two areas overlap.
- Since both points are in New York City (close to each other), the result is true.

If you change Location B to a far-away point like "Location C" (Los Angeles), the same query will return

You need to reproject your geometry into a meter-based coordinate system before applying ST_BUFFER. Use ST_TRANSFORM to convert to Web Mercator (EPSG:3857):

In [0]:
%sql
WITH transformed AS (
  SELECT
    name,
    ST_TRANSFORM(geo_point, 3857) AS geom_3857
  FROM locations
)
SELECT
  ST_INTERSECTS(
    ST_BUFFER((SELECT geom_3857 FROM transformed WHERE name='Location A'), 5000),
    ST_BUFFER((SELECT geom_3857 FROM transformed WHERE name='Location C'), 5000)
  ) AS do_overlap;



In [0]:
%sql
WITH transformed AS (
  SELECT
    name,
    ST_TRANSFORM(geo_point, 3857) AS geom_3857
  FROM locations
)
SELECT
  ST_INTERSECTS(
    ST_BUFFER((SELECT geom_3857 FROM transformed WHERE name='Location A'), 5000),
    ST_BUFFER((SELECT geom_3857 FROM transformed WHERE name='Location B'), 5000)
  ) AS do_overlap;


### 6. ST_Area

Compute area of a geometry (e.g., polygon, buffer).

In [0]:
from pyspark.sql.functions import expr

# Sample data: city name, latitude, longitude
data = [
    ("Location A", 40.748817, -73.985428),   # NYC
    ("Location B", 34.052235, -118.243683),  # LA
    ("Location C", 51.507222, -0.1275)       # London
]

df = spark.createDataFrame(data, ["name", "lat", "lon"])

# Convert to GEOMETRY type with SRID 4326 (WGS84 lon/lat)
df = df.withColumn("geo_point", expr("ST_SETSRID(ST_POINT(lon, lat), 4326)"))

df.createOrReplaceTempView("locations")
df.show(truncate=False)


Compute Buffer Area (Corrected for Meters)

To calculate the area in square kilometers, we first transform the geo_point to a projected coordinate system (EPSG:3857) where units are in meters, then buffer and measure the area:

In [0]:
%sql
WITH transformed AS (
  SELECT
    name,
    ST_TRANSFORM(geo_point, 3857) AS geom_3857   -- Transform to meter-based CRS
  FROM locations
)
SELECT
  name,
  ST_AREA(ST_BUFFER(geom_3857, 10000)) / 1e6 AS area_10km_sq_km  -- Convert m² to km²
FROM transformed;


Why We Use ST_TRANSFORM

- geo_point is in WGS84 (lat/lon) → unit is degrees, not meters.
- ST_BUFFER(geo_point, 10000) would incorrectly use 10,000 degrees.
- To buffer by 10,000 meters, we project the geometry to EPSG:3857

### 7. ST_Union

Combine two or more geometries into one.

In [0]:
from pyspark.sql.functions import expr

# Sample data: city name, latitude, longitude
data = [
    ("Location A", 40.748817, -73.985428),   # NYC
    ("Location B", 34.052235, -118.243683),  # LA
    ("Location C", 51.507222, -0.1275)       # London
]

df = spark.createDataFrame(data, ["name", "lat", "lon"])

# Convert to GEOMETRY with SRID 4326 (lon/lat)
df = df.withColumn("geo_point", expr("ST_SETSRID(ST_POINT(lon, lat), 4326)"))

df.createOrReplaceTempView("locations")
df.show(truncate=False)


In [0]:
%sql
SELECT 
  ST_UNION(a.geo_point, b.geo_point) AS unioned_geom
FROM locations a 
JOIN locations b
WHERE a.name = 'Location A' AND b.name = 'Location B';


How It Works

- ST_UNION(geom1, geom2) returns a geometry that represents the spatial union of inputs.
- For points, it creates a MULTIPOINT geometry.
- For overlapping polygons or buffers, ST_UNION would merge them into a single combined shape.

### ST_Transform

Transform one coordinate system to another.

In [0]:
%sql
SELECT 
  ST_TRANSFORM(geo_point, 3857) AS web_mercator
FROM locations;


### 9. ST_AsText / ST_AsGeoJSON

Convert spatial objects back into readable string formats (WKT, GeoJSON).

In [0]:
%sql
SELECT
  name,
  ST_ASTEXT(geo_point) AS wkt,
  ST_ASGEOJSON(geo_point) AS geojson
FROM locations;


### What It Does

- ST_ASTEXT()	Converts geo_point into WKT (Well-Known Text) format — human-readable geometry like POINT (-73.985428 40.748817)
- ST_ASGEOJSON()	Converts geo_point into GeoJSON format — useful for web mapping or APIs like: {"type":"Point","coordinates":[-73.985428,40.748817]}

Great for

- Debugging the stored geometry values
- Exporting spatial data to frontend map tools (Leaflet, Mapbox, etc.)
- Generating visualizations or interchange formats across systems

### 10. ST_Simplify

Simplify a geometry while retaining its general shape — very useful for maps.

In [0]:
%sql
SELECT
  name,
  ST_SIMPLIFY(ST_BUFFER(geo_point, 10000), 100) AS simplified
FROM locations;

![](/Volumes/thedataengineering_00/data/sampledata/ST_SheatSheet.png)

![](/Volumes/thedataengineering_00/data/sampledata/RealWorldUseCases.png)